## Execution Guide — Time-Block MobileNetV2 (Split v2)

This notebook is the official MobileNetV2 transfer-learning and experimentation workflow for split v2.

### Modes
Set these switches in the RESUME / SETUP cell:

- `MODE = "evaluate"` → load the saved final v2 MobileNetV2 checkpoint and evaluate without retraining.
- `MODE = "train"` → run Stage 1 and then Stage 2 as controlled below.
- `STAGE1_DONE = True` → skip Stage 1 after a runtime restart and restore its checkpoint by upload if it is not already under `/content`.
- `STAGE2_DONE = True` → skip Stage 2 and go directly to final evaluation using the Stage 2 checkpoint.

`MODE = "evaluate"` is the safe default. Do not use `Runtime → Run all`; run in order deliberately.

### Rules

- Use only the frozen official `data/time_block/` split v2.
- Do not change the 25-image block size or 60-second buffer.
- Training augmentation is applied only to the training dataset.
- Class weights are calculated from the training split only.
- Stage 1: frozen ImageNet MobileNetV2 backbone, learning rate `1e-3`.
- Stage 2: fine-tune only the selected top backbone layers, keep BatchNormalization frozen, learning rate `1e-5`.
- Never use old v0/v1 checkpoints for official v2 test results.
- Test results are produced only after the final Stage 2 checkpoint is fixed.
- Models, histories and evaluation artifacts stay under `/content/fruit_project/`. Use the final download cell to back them up locally.


# Fresh vs Rotten Fruit Classification — Time-Block MobileNetV2

## Objective
Train and evaluate MobileNetV2 using transfer learning and controlled fine-tuning on the official time-block split v2.

Six classes:
`freshapples`, `freshbanana`, `freshoranges`, `rottenapples`, `rottenbanana`, `rottenoranges`.

This notebook covers Onkar's MobileNetV2 transfer-learning and experimentation scope. Improved-CNN ablations and final project-wide comparison/prototype work are outside this notebook.


## 1. Official Experiment Setup

The official v2 split is frozen. The split manifest is the source of truth for image counts, source counts, class counts, block counts and the minimum cross-split time gap.

Results from v0 and v1 are not directly comparable with this v2 test result because their test sets are different.


In [ ]:
# =========================================================
# RESUME / SETUP
# Run this cell first after every fresh Colab runtime.
# =========================================================

import json
import os
import sys
import subprocess
from pathlib import Path

MODE = "evaluate"
STAGE1_DONE = False
STAGE2_DONE = False
RUN_SMOKE_TEST = False

REPO_URL = (
    "https://github.com/"
    "onkar-726/fresh-rotten-fruit-classification.git"
)
REPO_DIR = Path("/content/fresh-rotten-fruit-classification")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

subprocess.run([
    sys.executable, "-m", "pip", "install",
    "-r", "requirements-colab.txt", "-q"
], check=True)

import numpy as np
import pandas as pd
import tensorflow as tf
import kagglehub

from src.config import (
    CLASS_NAMES, NUM_CLASSES, IMAGE_SIZE, BATCH_SIZE, SEED,
    MIN_GAP_SECONDS, TRAIN_CSV, VALIDATION_CSV, TEST_CSV,
)
from src.colab_setup import set_seed, print_environment, verify_repo
from src.models import build_mobilenetv2, set_fine_tune

set_seed(SEED)

OUTPUT_ROOT = Path("/content/fruit_project")
MODEL_DIR = OUTPUT_ROOT / "models"
RESULTS_DIR = OUTPUT_ROOT / "results" / "mobilenetv2_time_block"
STAGE1_RESULTS_DIR = RESULTS_DIR / "stage1"
STAGE2_RESULTS_DIR = RESULTS_DIR / "stage2"
FULL_RESULTS_DIR = RESULTS_DIR / "full_test"
ORIGINAL_RESULTS_DIR = RESULTS_DIR / "original_only"
MISCLASSIFIED_DIR = RESULTS_DIR / "misclassified_test"

for folder in [MODEL_DIR, STAGE1_RESULTS_DIR, STAGE2_RESULTS_DIR,
               FULL_RESULTS_DIR, ORIGINAL_RESULTS_DIR, MISCLASSIFIED_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

verify_repo(REPO_DIR)

MANIFEST_PATH = REPO_DIR / "data" / "time_block" / "split_manifest.json"
if not MANIFEST_PATH.exists():
    raise FileNotFoundError(f"Missing official v2 manifest: {MANIFEST_PATH}")

with open(MANIFEST_PATH, "r", encoding="utf-8") as file:
    split_manifest = json.load(file)

assert split_manifest["split_version"] == "v2"
assert split_manifest["block_size"] == 25
assert split_manifest["buffer_seconds"] == MIN_GAP_SECONDS
assert split_manifest["closest_cross_split_gap_seconds"] >= MIN_GAP_SECONDS

dataset_download_path = Path(
    kagglehub.dataset_download(
        "sriramr/fruits-fresh-and-rotten-for-classification"
    )
)

dataset_candidates = [dataset_download_path, dataset_download_path / "dataset"]
if dataset_download_path.exists():
    dataset_candidates.extend(p for p in dataset_download_path.iterdir() if p.is_dir())

DATASET_ROOT = None
for candidate in dataset_candidates:
    if (candidate / "train").is_dir() and any((candidate / "train").iterdir()):
        DATASET_ROOT = candidate
        break

if DATASET_ROOT is None:
    raise FileNotFoundError("Could not find Kaggle dataset root containing train/.")

print("Repository:", REPO_DIR)
print("Dataset root:", DATASET_ROOT)
print_environment()
print("TensorFlow:", tf.__version__)
print("Python:", sys.version.split()[0])
print("Image size:", IMAGE_SIZE)
print("Batch size:", BATCH_SIZE)
print("Seed:", SEED)
print("Mode:", MODE)
print("Stage 1 done:", STAGE1_DONE)
print("Stage 2 done:", STAGE2_DONE)
print("Setup completed successfully.")


## 2. Load and Verify the Official v2 Split


In [ ]:
# =========================================================
# Load official time-block CSVs and verify integrity
# =========================================================

train_df = pd.read_csv(REPO_DIR / TRAIN_CSV)
validation_df = pd.read_csv(REPO_DIR / VALIDATION_CSV)
test_df = pd.read_csv(REPO_DIR / TEST_CSV)

expected_images = split_manifest["image_counts"]
expected_sources = split_manifest["source_counts"]

assert len(train_df) == expected_images["train"]
assert len(validation_df) == expected_images["validation"]
assert len(test_df) == expected_images["test"]
assert train_df["source_id"].nunique() == expected_sources["train"]
assert validation_df["source_id"].nunique() == expected_sources["validation"]
assert test_df["source_id"].nunique() == expected_sources["test"]

for name, df in [("train", train_df), ("validation", validation_df), ("test", test_df)]:
    assert set(df["class_name"]) == set(CLASS_NAMES), f"Missing class in {name}"

assert set(train_df.source_id).isdisjoint(validation_df.source_id)
assert set(train_df.source_id).isdisjoint(test_df.source_id)
assert set(validation_df.source_id).isdisjoint(test_df.source_id)

# Real capture-time verification, same class only.
combined = pd.concat([train_df, validation_df, test_df], ignore_index=True)
combined["capture_time"] = pd.to_datetime(combined["capture_time"], errors="raise")

closest_gap = None
for class_name in CLASS_NAMES:
    class_df = combined[combined["class_name"] == class_name]
    for a, b in [("train","validation"),("train","test"),("validation","test")]:
        left = class_df[class_df["split"] == a]["capture_time"]
        right = class_df[class_df["split"] == b]["capture_time"]
        if len(left) and len(right):
            # Compute closest pair without assuming equal row counts.
            lv = np.sort(left.values.astype("datetime64[s]").astype("int64"))
            rv = np.sort(right.values.astype("datetime64[s]").astype("int64"))
            j = 0
            for x in lv:
                while j + 1 < len(rv) and abs(rv[j+1] - x) <= abs(rv[j] - x):
                    j += 1
                gap = abs(rv[j] - x)
                closest_gap = gap if closest_gap is None else min(closest_gap, gap)

assert closest_gap >= MIN_GAP_SECONDS

print("Image counts:", {k: expected_images[k] for k in ["train","validation","test"]})
print("Source counts:", {k: expected_sources[k] for k in ["train","validation","test"]})
print("Minimum same-class cross-split time gap:", closest_gap, "seconds")
print("All classes present: PASSED")
print("Source overlap: 0")

print("\nBlocks per class:")
for split_name, df in [("train", train_df), ("validation", validation_df), ("test", test_df)]:
    print(split_name, df.groupby("class_name")["block_id"].nunique().to_dict())

print("Split verification: PASSED")


## 3. Input Pipeline + Sanity Batch

MobileNetV2 uses its own ImageNet preprocessing. Training augmentation is enabled only for the training dataset.


In [ ]:
# =========================================================
# TensorFlow datasets
# =========================================================

from src.data import build_dataset, compute_class_weights, original_only

train_ds = build_dataset(
    train_df, DATASET_ROOT, training=True, augment=True, mobilenet=True
)
validation_ds = build_dataset(
    validation_df, DATASET_ROOT, training=False, augment=False, mobilenet=True
)
test_ds = build_dataset(
    test_df, DATASET_ROOT, training=False, augment=False, mobilenet=True
)

# Class weights from training split only.
raw_weights = compute_class_weights(train_df)
if isinstance(raw_weights, dict):
    class_weights = {int(k): float(v) for k, v in raw_weights.items()}
else:
    class_weights = {i: float(raw_weights[i]) for i in range(NUM_CLASSES)}

print("Class weights:", class_weights)

for images, labels in train_ds.take(1):
    print("Batch image shape:", images.shape)
    print("Batch label shape:", labels.shape)
    print("Image dtype:", images.dtype)
    print("MobileNetV2 pixel range:", float(tf.reduce_min(images)), "to", float(tf.reduce_max(images)))
    assert images.shape[1:] == (*IMAGE_SIZE, 3)
    assert labels.shape[0] <= BATCH_SIZE
    assert float(tf.reduce_min(images)) >= -1.05
    assert float(tf.reduce_max(images)) <= 1.05

print("MobileNetV2 input pipeline check: PASSED")

if RUN_SMOKE_TEST:
    smoke_model, smoke_backbone = build_mobilenetv2()
    smoke_model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    smoke_history = smoke_model.fit(
        train_ds.take(2),
        validation_data=validation_ds.take(1),
        epochs=1,
        class_weight=class_weights,
        verbose=1,
    )
    print("Smoke-test accuracy:", smoke_history.history["accuracy"][-1])
    print("Smoke test: PASSED")
else:
    print("RUN_SMOKE_TEST=False — skipped.")


## 4. Model Definition

The model is created through `src/models.py`. The ImageNet MobileNetV2 backbone is frozen initially and a six-class softmax head is trained.


In [ ]:
# =========================================================
# Build MobileNetV2
# =========================================================

model, backbone = build_mobilenetv2()

model.summary()
print("Total parameters:", model.count_params())
print("Trainable parameters:", sum(int(np.prod(v.shape)) for v in model.trainable_weights))


## 5. Stage 1 — Frozen-Backbone Transfer Learning

Only the new classification head is trained.


In [ ]:
# =========================================================
# Stage 1 training
# =========================================================

STAGE1_MODEL_NAME = "mobilenetv2_stage1_time_block_best.keras"
STAGE1_MODEL_PATH = MODEL_DIR / STAGE1_MODEL_NAME
STAGE1_HISTORY_CSV = STAGE1_RESULTS_DIR / "mobilenetv2_stage1_history.csv"
STAGE1_HISTORY_JSON = STAGE1_RESULTS_DIR / "mobilenetv2_stage1_history.json"

STAGE1_EPOCHS = 15
STAGE1_LR = 1e-3

if MODE == "train" and not STAGE1_DONE:
    tf.keras.backend.clear_session()
    model, backbone = build_mobilenetv2()

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=STAGE1_LR),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    stage1_callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=3, restore_best_weights=True, verbose=1
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6, verbose=1
        ),
        tf.keras.callbacks.ModelCheckpoint(
            str(STAGE1_MODEL_PATH), monitor="val_loss", save_best_only=True, verbose=1
        ),
    ]

    print("Stage 1 training...")
    stage1_history = model.fit(
        train_ds,
        validation_data=validation_ds,
        epochs=STAGE1_EPOCHS,
        class_weight=class_weights,
        callbacks=stage1_callbacks,
        verbose=1,
    )

    stage1_data = {k: [float(x) for x in v] for k, v in stage1_history.history.items()}
    pd.DataFrame(stage1_data).to_csv(STAGE1_HISTORY_CSV, index=False)
    with open(STAGE1_HISTORY_JSON, "w", encoding="utf-8") as f:
        json.dump(stage1_data, f, indent=2)

    from src.evaluate import plot_history
    plot_history(stage1_history, output_dir=STAGE1_RESULTS_DIR, prefix="mobilenetv2_stage1_time_block")

    best_epoch = min(range(len(stage1_data["val_loss"])), key=lambda i: stage1_data["val_loss"][i]) + 1
    print("Stage 1 epochs:", len(stage1_data["loss"]))
    print("Stage 1 best epoch:", best_epoch)
    print("Stage 1 best val loss:", stage1_data["val_loss"][best_epoch - 1])
    print("Stage 1 best val accuracy:", stage1_data["val_accuracy"][best_epoch - 1])

    assert STAGE1_MODEL_PATH.exists()
    assert STAGE1_HISTORY_CSV.exists()
    assert STAGE1_HISTORY_JSON.exists()

elif MODE == "train" and STAGE1_DONE:
    print("STAGE1_DONE=True — Stage 1 skipped.")
else:
    print("MODE='evaluate' — Stage 1 training not run.")


## 6. Stage 2 — Controlled Fine-Tuning

Reload the Stage 1 checkpoint, unfreeze only the selected top backbone layers, keep all BatchNormalization layers frozen, and use the smaller learning rate `1e-5`.


In [ ]:
# =========================================================
# Prepare / restore Stage 1 model before Stage 2
# =========================================================

def get_or_upload_checkpoint(expected_name, local_path):
    from pathlib import Path

    if Path(local_path).exists():
        print("Using local checkpoint:", local_path)
        return Path(local_path)

    from google.colab import files
    print("Checkpoint not found locally. Upload:", expected_name)
    uploaded = files.upload()
    assert len(uploaded) == 1, "Upload exactly one checkpoint file."
    uploaded_name = next(iter(uploaded))
    assert uploaded_name == expected_name, f"Expected {expected_name}, got {uploaded_name}"
    return Path("/content") / uploaded_name

if MODE == "train" and not STAGE2_DONE:
    stage1_checkpoint = get_or_upload_checkpoint(STAGE1_MODEL_NAME, STAGE1_MODEL_PATH)
    model = tf.keras.models.load_model(stage1_checkpoint)

    # Find the MobileNetV2 backbone by layer type rather than by an auto-generated name.
    backbone_candidates = [layer for layer in model.layers if isinstance(layer, tf.keras.Model)]
    assert backbone_candidates, "MobileNetV2 backbone could not be found."
    backbone = next(
        layer for layer in backbone_candidates
        if isinstance(layer, tf.keras.Model) and "MobileNetV2" in layer.__class__.__name__
        or "mobilenetv2" in layer.name.lower()
    )

    trainable_count, weight_trainable_count = set_fine_tune(
        model, backbone, n_layers=30
    )

    assert all(
        not layer.trainable
        for layer in backbone.layers
        if isinstance(layer, tf.keras.layers.BatchNormalization)
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    model.summary()
    print("Total parameters:", model.count_params())
    print("Trainable backbone layers:", trainable_count)
    print("Trainable weight-bearing backbone layers:", weight_trainable_count)
    print("BatchNormalization layers trainable: 0")

else:
    print("Stage 2 preparation deferred until training/evaluation checkpoint loading.")


In [ ]:
# =========================================================
# Stage 2 training
# =========================================================

STAGE2_MODEL_NAME = "mobilenetv2_time_block_best.keras"
STAGE2_MODEL_PATH = MODEL_DIR / STAGE2_MODEL_NAME
STAGE2_HISTORY_CSV = STAGE2_RESULTS_DIR / "mobilenetv2_time_block_history.csv"
STAGE2_HISTORY_JSON = STAGE2_RESULTS_DIR / "mobilenetv2_time_block_history.json"

STAGE2_EPOCHS = 15
STAGE2_LR = 1e-5

if MODE == "train" and not STAGE2_DONE:
    stage2_callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=3, restore_best_weights=True, verbose=1
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=2, min_lr=1e-7, verbose=1
        ),
        tf.keras.callbacks.ModelCheckpoint(
            str(STAGE2_MODEL_PATH), monitor="val_loss", save_best_only=True, verbose=1
        ),
    ]

    print("Stage 2 fine-tuning...")
    stage2_history = model.fit(
        train_ds,
        validation_data=validation_ds,
        epochs=STAGE2_EPOCHS,
        class_weight=class_weights,
        callbacks=stage2_callbacks,
        verbose=1,
    )

    stage2_data = {k: [float(x) for x in v] for k, v in stage2_history.history.items()}
    pd.DataFrame(stage2_data).to_csv(STAGE2_HISTORY_CSV, index=False)
    with open(STAGE2_HISTORY_JSON, "w", encoding="utf-8") as f:
        json.dump(stage2_data, f, indent=2)

    from src.evaluate import plot_history
    plot_history(stage2_history, output_dir=STAGE2_RESULTS_DIR, prefix="mobilenetv2_time_block")

    best_epoch = min(range(len(stage2_data["val_loss"])), key=lambda i: stage2_data["val_loss"][i]) + 1
    print("Stage 2 epochs:", len(stage2_data["loss"]))
    print("Stage 2 best epoch:", best_epoch)
    print("Stage 2 best val loss:", stage2_data["val_loss"][best_epoch - 1])
    print("Stage 2 best val accuracy:", stage2_data["val_accuracy"][best_epoch - 1])

    assert STAGE2_MODEL_PATH.exists()
    assert STAGE2_HISTORY_CSV.exists()
    assert STAGE2_HISTORY_JSON.exists()

elif MODE == "train" and STAGE2_DONE:
    print("STAGE2_DONE=True — Stage 2 skipped.")
else:
    print("MODE='evaluate' — Stage 2 training not run.")


## 7. Load the Final Saved MobileNetV2

This cell never retrains. It loads the local Stage 2 checkpoint or asks you to upload the exact uniquely named checkpoint.


In [ ]:
# =========================================================
# Load final MobileNetV2 checkpoint
# =========================================================

EXPECTED_FINAL_NAME = "mobilenetv2_time_block_best.keras"
LOCAL_FINAL_PATH = MODEL_DIR / EXPECTED_FINAL_NAME

if LOCAL_FINAL_PATH.exists():
    FINAL_MODEL_PATH = LOCAL_FINAL_PATH
    print("Using local final checkpoint:", FINAL_MODEL_PATH)
else:
    from google.colab import files
    print("Local final checkpoint not found. Upload:", EXPECTED_FINAL_NAME)
    uploaded = files.upload()
    assert len(uploaded) == 1, "Upload exactly one .keras model file."
    uploaded_name = next(iter(uploaded))
    assert uploaded_name == EXPECTED_FINAL_NAME, (
        f"Wrong model file: {uploaded_name}. Expected: {EXPECTED_FINAL_NAME}"
    )
    FINAL_MODEL_PATH = Path("/content") / uploaded_name

assert FINAL_MODEL_PATH.exists()
assert FINAL_MODEL_PATH.name == EXPECTED_FINAL_NAME

model = tf.keras.models.load_model(FINAL_MODEL_PATH)
print("Loaded:", FINAL_MODEL_PATH)
print("Model name:", model.name)
print("Parameters:", model.count_params())


## 8. Full Time-Block Test Evaluation


In [ ]:
# =========================================================
# Full test evaluation and export
# =========================================================

from src.evaluate import evaluate_model, export_results

full_test_loss, full_test_accuracy = model.evaluate(test_ds, verbose=1)
y_true, y_pred, y_prob = evaluate_model(model, test_ds)

assert len(y_true) == len(test_df)
assert len(y_pred) == len(test_df)
assert y_prob.shape[0] == len(test_df)

full_summary = export_results(
    y_true=y_true, y_pred=y_pred, probabilities=y_prob, class_names=CLASS_NAMES,
    output_dir=FULL_RESULTS_DIR,
    prefix="mobilenetv2_time_block_full",
    metadata={
        "split": "time_block_v2", "test_view": "full",
        "test_images": len(test_df),
        "test_sources": int(test_df["source_id"].nunique()),
        "test_loss": float(full_test_loss),
        "test_accuracy": float(full_test_accuracy),
        "parameters": int(model.count_params()),
        "tensorflow_version": tf.__version__, "seed": SEED,
    },
)

full_predictions = test_df[
    ["image_path", "class_name", "source_filename", "source_id", "capture_time", "block_id"]
].reset_index(drop=True).copy()
full_predictions["true_label"] = y_true
full_predictions["predicted_label"] = y_pred
full_predictions["true_class"] = [CLASS_NAMES[i] for i in y_true]
full_predictions["predicted_class"] = [CLASS_NAMES[i] for i in y_pred]
full_predictions["confidence"] = np.max(y_prob, axis=1)

full_predictions.to_csv(
    FULL_RESULTS_DIR / "mobilenetv2_time_block_full_predictions.csv",
    index=False,
)

print("\nFull test results")
print("Loss:", full_test_loss)
print("Accuracy:", full_test_accuracy)
print("Macro F1:", full_summary["macro_f1"])
print("Weighted F1:", full_summary["weighted_f1"])


## 9. Original-Only Test Evaluation

This view keeps exactly one non-augmented original image per source photo.


In [ ]:
# =========================================================
# Original-only evaluation and export
# =========================================================

original_test_df = original_only(test_df)
assert len(original_test_df) == original_test_df["source_id"].nunique()
assert set(original_test_df["class_name"]) == set(CLASS_NAMES)

original_test_ds = build_dataset(
    original_test_df, DATASET_ROOT, training=False, augment=False, mobilenet=True
)

original_test_loss, original_test_accuracy = model.evaluate(original_test_ds, verbose=1)
original_y_true, original_y_pred, original_y_prob = evaluate_model(model, original_test_ds)

assert len(original_y_true) == len(original_test_df)
assert len(original_y_pred) == len(original_test_df)

original_summary = export_results(
    y_true=original_y_true, y_pred=original_y_pred, probabilities=original_y_prob,
    class_names=CLASS_NAMES, output_dir=ORIGINAL_RESULTS_DIR,
    prefix="mobilenetv2_time_block_original_only",
    metadata={
        "split": "time_block_v2", "test_view": "original_only",
        "test_images": len(original_test_df),
        "test_sources": int(original_test_df["source_id"].nunique()),
        "test_loss": float(original_test_loss),
        "test_accuracy": float(original_test_accuracy),
        "parameters": int(model.count_params()),
        "tensorflow_version": tf.__version__, "seed": SEED,
    },
)

original_predictions = original_test_df[
    ["image_path", "class_name", "source_filename", "source_id", "capture_time", "block_id"]
].reset_index(drop=True).copy()
original_predictions["true_label"] = original_y_true
original_predictions["predicted_label"] = original_y_pred
original_predictions["true_class"] = [CLASS_NAMES[i] for i in original_y_true]
original_predictions["predicted_class"] = [CLASS_NAMES[i] for i in original_y_pred]
original_predictions["confidence"] = np.max(original_y_prob, axis=1)
original_predictions.to_csv(
    ORIGINAL_RESULTS_DIR / "mobilenetv2_time_block_original_only_predictions.csv",
    index=False,
)

print("\nOriginal-only results")
print("Images:", len(original_test_df))
print("Loss:", original_test_loss)
print("Accuracy:", original_test_accuracy)
print("Macro F1:", original_summary["macro_f1"])
print("Weighted F1:", original_summary["weighted_f1"])


## 10. Top 12 Misclassified Test Images


In [ ]:
# =========================================================
# Top 12 misclassified images
# =========================================================

import matplotlib.pyplot as plt
from IPython.display import display

misclassified_mask = y_true != y_pred
misclassified = full_predictions.loc[
    misclassified_mask,
    ["image_path", "true_class", "predicted_class", "confidence", "source_id"],
].sort_values("confidence", ascending=False).head(12).reset_index(drop=True)

misclassified.to_csv(
    MISCLASSIFIED_DIR / "mobilenetv2_time_block_top12_misclassified.csv",
    index=False,
)

print("Misclassified test images:", int(misclassified_mask.sum()))
display(misclassified)

if len(misclassified):
    fig, axes = plt.subplots(3, 4, figsize=(14, 10))
    axes = axes.ravel()
    for ax in axes:
        ax.axis("off")

    for i in range(min(12, len(misclassified))):
        row = misclassified.iloc[i]
        image_path = DATASET_ROOT / str(row["image_path"])
        if image_path.exists():
            axes[i].imshow(plt.imread(str(image_path)))
            axes[i].set_title(
                f"True: {row['true_class']}\nPred: {row['predicted_class']}\nConf: {row['confidence']:.3f}"
            )
        else:
            axes[i].text(0.5, 0.5, "Image not found", ha="center", va="center")
        axes[i].axis("off")

    plt.tight_layout()
    path = MISCLASSIFIED_DIR / "mobilenetv2_time_block_top12_misclassified.png"
    plt.savefig(path, dpi=200, bbox_inches="tight")
    plt.show()
    print("Saved:", path)
else:
    print("No misclassified images found.")


## 11. Experiment Metadata and Repository Result Export


In [ ]:
# =========================================================
# Metadata + small result artifacts for Git repository
# =========================================================

import shutil

metadata = {
    "experiment": "mobilenetv2_time_block",
    "split": "time_block_v2",
    "stage1_checkpoint": STAGE1_MODEL_NAME,
    "stage2_checkpoint": STAGE2_MODEL_NAME,
    "parameters": int(model.count_params()),
    "image_size": list(IMAGE_SIZE),
    "batch_size": BATCH_SIZE,
    "seed": SEED,
    "stage1_learning_rate": STAGE1_LR,
    "stage2_learning_rate": STAGE2_LR,
    "stage2_unfrozen_backbone_layers_target": 30,
    "batchnorm_trainable": False,
    "training_augmentation": True,
    "class_weights": True,
    "test_images": int(len(test_df)),
    "test_sources": int(test_df["source_id"].nunique()),
    "test_loss": float(full_test_loss),
    "test_accuracy": float(full_test_accuracy),
    "test_macro_f1": float(full_summary["macro_f1"]),
    "test_weighted_f1": float(full_summary["weighted_f1"]),
    "original_only_images": int(len(original_test_df)),
    "original_only_accuracy": float(original_test_accuracy),
    "original_only_macro_f1": float(original_summary["macro_f1"]),
    "tensorflow_version": tf.__version__,
}

metadata_path = RESULTS_DIR / "mobilenetv2_time_block_experiment_metadata.json"
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

REPO_RESULTS_DIR = REPO_DIR / "results" / "time_block" / "mobilenetv2"
REPO_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

source_files = []
for folder in [STAGE1_RESULTS_DIR, STAGE2_RESULTS_DIR, FULL_RESULTS_DIR, ORIGINAL_RESULTS_DIR, MISCLASSIFIED_DIR]:
    if folder.exists():
        source_files.extend(p for p in folder.rglob("*") if p.is_file())
source_files.append(metadata_path)

for source in source_files:
    shutil.copy2(source, REPO_RESULTS_DIR / source.name)

assert not any(
    p.suffix.lower() in {".keras", ".h5"}
    for p in REPO_RESULTS_DIR.iterdir()
)

print("Repository result directory:", REPO_RESULTS_DIR)
print("Result artifact copy: completed")


## 12. Download / Backup Before Closing Colab

Colab `/content` storage is temporary. Download the final checkpoint and result ZIP before ending the runtime.


In [ ]:
# =========================================================
# Final local backup
# =========================================================

from google.colab import files
import shutil

archive_path = shutil.make_archive(
    "/content/mobilenetv2_time_block_results",
    "zip",
    RESULTS_DIR.parent,
    RESULTS_DIR.name,
)

print("Downloading results ZIP:", archive_path)
files.download(archive_path)

final_model_path = MODEL_DIR / EXPECTED_FINAL_NAME
if final_model_path.exists():
    print("Downloading final model:", final_model_path)
    files.download(str(final_model_path))
else:
    print("Final model not found:", final_model_path)

stage1_path = MODEL_DIR / STAGE1_MODEL_NAME
if stage1_path.exists():
    print("Stage 1 checkpoint also exists locally:", stage1_path)
    print("Download it separately only if you need Stage 2 restart capability.")

print("Backup step completed.")


## 13. Final Status and Limitations

This notebook provides the official v2 MobileNetV2 transfer-learning experiment, Stage 1 and Stage 2 histories, full and original-only test results, predictions, confusion matrices, classification reports, and misclassification artifacts.

The official v2 test set is small at the capture-block level, so performance should be interpreted with the saved prediction files and later uncertainty analysis. Do not compare these numbers with v0/v1 numbers as though they came from the same test set.

Before closing Colab, download the final `.keras` checkpoint and the MobileNetV2 result ZIP.
